# B2B Pricing — Historical Date Diagnostic

**Purpose**: Verify whether the current code and the `Analytics_Rpt.md.B2BPricingHist` table support running the B2B report as of a specific historical date (e.g., **Jan 31, 2026**).

This notebook checks:
1. What `AsofDt` snapshots are available in `B2BPricingHist`
2. Loan counts and UPB in the Jan-2026 funding window (`FundDt Feb 1, 2025 → Jan 31, 2026`)
3. What performance rows exist in `B2BPricingHist` for those loans
4. How many months of score data exist per loan — understanding what CPR / Recap calculations would see
5. The date parameter impact — showing what changes between a current run vs. a Jan-2026 historical run

Use the **Python (B2B)** kernel (env: `B2B`).

## 0. Parameters — Set your target historical date here

In [ ]:
from datetime import date

# ── Target historical report date ────────────────────────────────────────────
REPORT_AS_OF_DT = date(2026, 1, 31)

# Derived funding window (mirrors the stored procedure default logic)
# For a Jan-2026 report run in early Feb-2026:
#   begin_fund_dt = first day of the month 13 months prior to REPORT_AS_OF_DT
#   end_fund_dt   = REPORT_AS_OF_DT itself
import calendar
from datetime import timedelta

def eomonth(d, offset=0):
    year  = d.year + (d.month - 1 + offset) // 12
    month = (d.month - 1 + offset) % 12 + 1
    return date(year, month, calendar.monthrange(year, month)[1])

BEGIN_FUND_DT = eomonth(REPORT_AS_OF_DT, -13) + timedelta(days=1)  # Feb 1, 2025
END_FUND_DT   = REPORT_AS_OF_DT                                     # Jan 31, 2026
END_SCORE_DT  = REPORT_AS_OF_DT                                     # Jan 31, 2026

# What the CURRENT run (today) would use:
TODAY = date.today()
CURRENT_END_FUND_DT   = eomonth(TODAY, -1)
CURRENT_BEGIN_FUND_DT = eomonth(TODAY, -13) + timedelta(days=1)

print('=' * 60)
print(f'HISTORICAL report as of : {REPORT_AS_OF_DT}')
print(f'  Funding window         : {BEGIN_FUND_DT}  →  {END_FUND_DT}')
print(f'  Score date cap         : up to {END_SCORE_DT}')
print()
print(f'CURRENT run (today {TODAY}):')
print(f'  Funding window         : {CURRENT_BEGIN_FUND_DT}  →  {CURRENT_END_FUND_DT}')
print(f'  Score date cap         : ALL history (no cap)')
print('=' * 60)

## 1. Imports & SQL Connection

In [ ]:
import os, sys
import pandas as pd
import numpy as np
import sqlalchemy
from sqlalchemy import text

FUNCTIONS_DIR = os.path.join(os.getcwd(), 'functions')
if FUNCTIONS_DIR not in sys.path:
    sys.path.insert(0, FUNCTIONS_DIR)

from B2B_functions_monthly_report import sql_connection, DateHelpers

engine = sql_connection(
    server='VMC2VMDBRADC01P',
    database='Analytics_Rpt',
    driver='ODBC Driver 17 for SQL Server',
)
print('Connected:', engine)

## 2. Check 1: Available `AsofDt` snapshots in `B2BPricingHist`

This tells us what months exist, how many loan-month rows are in each snapshot, and whether Jan 2026 data is present.

In [ ]:
q = text("""
SELECT
    AsofDt,
    LoanCount      = COUNT(DISTINCT LoanNum),
    RowCount       = COUNT(*),
    TotalUPB       = SUM(BeginPrinBal),
    MinFundedAmt   = MIN(FundedLoanAmt),
    MaxFundedAmt   = MAX(FundedLoanAmt),
    AvgModelCPR    = AVG(ModelCPR),
    HasCohortRecap = SUM(CASE WHEN CohortRecap IS NOT NULL THEN 1 ELSE 0 END)
FROM Analytics_Rpt.md.B2BPricingHist
WHERE FundedLoanAmt > 0
GROUP BY AsofDt
ORDER BY AsofDt
""")

df_snapshots = pd.read_sql(q, engine)
df_snapshots['AsofDt'] = pd.to_datetime(df_snapshots['AsofDt'])

print(f'Total monthly snapshots available: {len(df_snapshots)}')
print(f'Date range: {df_snapshots.AsofDt.min().date()}  to  {df_snapshots.AsofDt.max().date()}')
print(f'Jan 2026 present: {date(2026, 1, 31) in df_snapshots.AsofDt.dt.date.values}')
print()

# Highlight Jan 2026 and surrounding months
highlight = df_snapshots[
    df_snapshots['AsofDt'].between('2025-10-01', '2026-04-30')
].copy()
highlight['TotalUPB'] = highlight['TotalUPB'].apply(lambda x: f'${x:,.0f}' if pd.notna(x) else '')
print('--- Snapshots around Jan 2026 ---')
highlight

## 3. Check 2: Volume — Loans funded in the Jan-2026 window

Funding window: **Feb 1, 2025 → Jan 31, 2026** (same 13-month window the stored procedure would use when run in early Feb 2026).

In [ ]:
q_vol = text("""
SELECT
    FundMonth      = EOMONTH(FundDt),
    LoanCount      = COUNT(DISTINCT LoanNum),
    FundedLoanAmt  = SUM(FundedLoanAmt)
FROM ADM.ODS.vwOrigLoan a
WHERE a.Channel IN ('Correspondent', 'Co-Issue')
  AND a.Client IS NOT NULL
  AND a.FundDt BETWEEN :begin_dt AND :end_dt
  AND a.FundedLoanAmt > 0
  AND a.MortgageType <> 'Reverse'
  AND NOT EXISTS (
      SELECT 1 FROM Analytics_Rpt.md.B2BPricingHist h
      WHERE h.LoanNum = a.LoanNum
        AND (h.ResolType = 'XFER' OR h.Channel = 'Bulk')
  )
GROUP BY EOMONTH(FundDt)
ORDER BY EOMONTH(FundDt)
""")

df_vol = pd.read_sql(q_vol, engine, params={'begin_dt': BEGIN_FUND_DT, 'end_dt': END_FUND_DT})
df_vol['FundMonth'] = pd.to_datetime(df_vol['FundMonth']).dt.date
df_vol['FundedLoanAmt'] = df_vol['FundedLoanAmt'].apply(lambda x: f'${x:,.0f}')

total_loans = df_vol['LoanCount'].sum()
print(f'Funding window : {BEGIN_FUND_DT}  →  {END_FUND_DT}')
print(f'Total unique loans in window: {total_loans:,}')
print()
df_vol

## 4. Check 3: Performance — `B2BPricingHist` rows for those loans

For loans in the funding window, how many `B2BPricingHist` snapshot months exist — and critically, how many exist **after** Jan 31, 2026? Those are the rows that would contaminate a historical report if `pi_EndScoreDt` is left as `None`.

In [ ]:
q_perf = text("""
SELECT
    h.AsofDt,
    LoanCount     = COUNT(DISTINCT h.LoanNum),
    RowCount      = COUNT(*),
    TotalBeginBal = SUM(h.BeginPrinBal),
    TotalEndBal   = SUM(h.EndPrinBal),
    AvgModelCPR   = AVG(h.ModelCPR),
    PIF_loans     = SUM(CASE WHEN h.PIFBal > 0 THEN 1 ELSE 0 END),
    Recap_loans   = SUM(CASE WHEN h.RecapBal > 0 THEN 1 ELSE 0 END)
FROM Analytics_Rpt.md.B2BPricingHist h
INNER JOIN ADM.ODS.vwOrigLoan o
    ON h.LoanNum = o.LoanNum
WHERE h.FundedLoanAmt > 0
  AND o.Channel IN ('Correspondent', 'Co-Issue')
  AND o.Client IS NOT NULL
  AND o.FundDt BETWEEN :begin_dt AND :end_dt
  AND o.FundedLoanAmt > 0
  AND o.MortgageType <> 'Reverse'
  AND NOT EXISTS (
      SELECT 1 FROM Analytics_Rpt.md.B2BPricingHist x
      WHERE x.LoanNum = h.LoanNum
        AND (x.ResolType = 'XFER' OR x.Channel = 'Bulk')
  )
GROUP BY h.AsofDt
ORDER BY h.AsofDt
""")

df_perf = pd.read_sql(q_perf, engine, params={'begin_dt': BEGIN_FUND_DT, 'end_dt': END_FUND_DT})
df_perf['AsofDt'] = pd.to_datetime(df_perf['AsofDt']).dt.date
df_perf['in_historical_window'] = df_perf['AsofDt'] <= END_SCORE_DT

before = df_perf[df_perf['in_historical_window']]
after  = df_perf[~df_perf['in_historical_window']]

print(f'Score months UP TO Jan 31, 2026   : {len(before)} months, {before["LoanCount"].sum():,} loan-month rows')
print(f'Score months AFTER Jan 31, 2026   : {len(after)} months, {after["LoanCount"].sum():,} loan-month rows')
print()
print('If pi_EndScoreDt = None (current default), those AFTER rows get included in CPR/Recap!')
print()
df_perf

## 5. Check 4: CPR Impact — What changes when you cap `EndScoreDt` at Jan 31, 2026?

Compute `ActualCPR` and `ModelCPR` for a **Total** row under two scenarios:
- **A) Uncapped** — score dates = all history (current default)
- **B) Capped** — score dates capped at Jan 31, 2026 (correct for historical report)

In [ ]:
q_cpr = text("""
SELECT
    Scenario            = :scenario_label,
    AsofDtFrom          = MIN(h.AsofDt),
    AsofDtTo            = MAX(h.AsofDt),
    MonthsOfData        = COUNT(DISTINCT h.AsofDt),
    TotalBeginBal       = SUM(h.BeginPrinBal),
    TotalEndBal         = SUM(h.EndPrinBal),
    ActualCPR_formula   = 1.0 - POWER(SUM(h.EndPrinBal) * 1.0 / NULLIF(SUM(h.BeginPrinBal), 0), 12),
    ModelCPR_wavg       = SUM(h.ModelCPR * h.BeginPrinBal) / NULLIF(SUM(h.BeginPrinBal), 0),
    PIF_loans           = SUM(CASE WHEN h.PIFBal > 0 THEN 1 ELSE 0 END),
    TotalRecapBal       = SUM(h.RecapBal),
    TotalPIFBal         = SUM(CASE WHEN h.PIFReason IN ('Purchase','Rate/Term Refinance','Cash-Out Refinance')
                                   THEN h.PIFBal ELSE 0 END),
    ActualRecap         = SUM(h.RecapBal) / NULLIF(
                            SUM(CASE WHEN h.PIFReason IN ('Purchase','Rate/Term Refinance','Cash-Out Refinance')
                                     THEN h.PIFBal ELSE 0 END), 0)
FROM Analytics_Rpt.md.B2BPricingHist h
INNER JOIN ADM.ODS.vwOrigLoan o ON h.LoanNum = o.LoanNum
WHERE h.FundedLoanAmt > 0
  AND h.AsofDt BETWEEN :score_begin AND :score_end
  AND o.Channel IN ('Correspondent', 'Co-Issue')
  AND o.Client IS NOT NULL
  AND o.FundDt BETWEEN :fund_begin AND :fund_end
  AND o.FundedLoanAmt > 0
  AND o.MortgageType <> 'Reverse'
  AND NOT EXISTS (
      SELECT 1 FROM Analytics_Rpt.md.B2BPricingHist x
      WHERE x.LoanNum = h.LoanNum
        AND (x.ResolType = 'XFER' OR x.Channel = 'Bulk')
  )
""")

rows = []

# Scenario A: Uncapped (all history)
rows.append(pd.read_sql(q_cpr, engine, params={
    'scenario_label': 'A: Uncapped (all history)',
    'score_begin': date(1900, 1, 1), 'score_end': date(9999, 12, 31),
    'fund_begin': BEGIN_FUND_DT, 'fund_end': END_FUND_DT,
}))

# Scenario B: Capped at Jan 31, 2026
rows.append(pd.read_sql(q_cpr, engine, params={
    'scenario_label': f'B: Capped at {END_SCORE_DT}',
    'score_begin': date(1900, 1, 1), 'score_end': END_SCORE_DT,
    'fund_begin': BEGIN_FUND_DT, 'fund_end': END_FUND_DT,
}))

df_cpr_compare = pd.concat(rows, ignore_index=True)
for col in ['TotalBeginBal', 'TotalEndBal', 'TotalRecapBal', 'TotalPIFBal']:
    df_cpr_compare[col] = df_cpr_compare[col].apply(lambda x: f'${x:,.0f}' if pd.notna(x) else '')
for col in ['ActualCPR_formula', 'ModelCPR_wavg', 'ActualRecap']:
    df_cpr_compare[col] = df_cpr_compare[col].apply(lambda x: f'{x:.4%}' if pd.notna(x) else 'N/A')

print('CPR & Recap comparison — Uncapped vs Capped at Jan 31, 2026:')
df_cpr_compare.T

## 6. Check 5: Per-loan score month count distribution

Understand how many months of `B2BPricingHist` rows the CPR formula sees per loan in the Jan-2026 funding cohort. The `ActualCPR` formula `1 - (ΣEndBal/ΣBeginBal)^12` is meaningful only when the denominator and numerator are accumulated over a full 12-month window.

In [ ]:
q_months = text("""
SELECT
    ScoreMonthCount  = COUNT(DISTINCT h.AsofDt),
    NumLoans         = COUNT(DISTINCT h.LoanNum)
FROM Analytics_Rpt.md.B2BPricingHist h
INNER JOIN ADM.ODS.vwOrigLoan o ON h.LoanNum = o.LoanNum
WHERE h.FundedLoanAmt > 0
  AND h.AsofDt <= :score_end
  AND o.Channel IN ('Correspondent', 'Co-Issue')
  AND o.Client IS NOT NULL
  AND o.FundDt BETWEEN :fund_begin AND :fund_end
  AND o.FundedLoanAmt > 0
  AND o.MortgageType <> 'Reverse'
  AND NOT EXISTS (
      SELECT 1 FROM Analytics_Rpt.md.B2BPricingHist x
      WHERE x.LoanNum = h.LoanNum
        AND (x.ResolType = 'XFER' OR x.Channel = 'Bulk')
  )
GROUP BY h.LoanNum
""")

# We need to group the above — wrap it
q_dist = text("""
WITH loan_months AS (
    SELECT
        h.LoanNum,
        ScoreMonthCount = COUNT(DISTINCT h.AsofDt)
    FROM Analytics_Rpt.md.B2BPricingHist h
    INNER JOIN ADM.ODS.vwOrigLoan o ON h.LoanNum = o.LoanNum
    WHERE h.FundedLoanAmt > 0
      AND h.AsofDt <= :score_end
      AND o.Channel IN ('Correspondent', 'Co-Issue')
      AND o.Client IS NOT NULL
      AND o.FundDt BETWEEN :fund_begin AND :fund_end
      AND o.FundedLoanAmt > 0
      AND o.MortgageType <> 'Reverse'
      AND NOT EXISTS (
          SELECT 1 FROM Analytics_Rpt.md.B2BPricingHist x
          WHERE x.LoanNum = h.LoanNum
            AND (x.ResolType = 'XFER' OR x.Channel = 'Bulk')
      )
    GROUP BY h.LoanNum
)
SELECT
    ScoreMonthCount,
    NumLoans = COUNT(*),
    PctLoans = CAST(COUNT(*) AS FLOAT) / SUM(COUNT(*)) OVER ()
FROM loan_months
GROUP BY ScoreMonthCount
ORDER BY ScoreMonthCount
""")

df_dist = pd.read_sql(q_dist, engine, params={
    'score_end': END_SCORE_DT,
    'fund_begin': BEGIN_FUND_DT, 'fund_end': END_FUND_DT,
})
df_dist['PctLoans'] = df_dist['PctLoans'].apply(lambda x: f'{x:.1%}')

print(f'Distribution of score months per loan (capped at {END_SCORE_DT})')
print('(Jan-2026 funding cohort: Feb 2025 → Jan 2026 funded loans)')
print()
print('ScoreMonthCount = 1 means the loan only has 1 month of B2BPricingHist data → ActualCPR is annualized single-month SMM')
print('ScoreMonthCount = 12 means the loan has all 12 months → more stable CPR estimate')
print()
df_dist

## 7. Check 6: Exact parameter values for running a Jan-2026 historical report

This shows the exact values to paste into `B2B_Reporting_monthly_report.ipynb` cell 6 (Parameters).

In [ ]:
from datetime import date

print('=' * 65)
print('PARAMETERS TO USE IN B2B_Reporting_monthly_report.ipynb')
print('For historical report as of: Jan 31, 2026')
print('=' * 65)
print()
print('# Cell 6 — Parameters')
print(f'pi_Product        = \'All\'')
print(f'pi_BeginScoreDt   = None          # All history through EndScoreDt')
print(f'pi_EndScoreDt     = date(2026, 1, 31)  # <-- MUST SET for historical run')
print(f'pi_BeginRefFundDt = date(2025, 2, 1)   # 13-month window start')
print(f'pi_EndRefFundDt   = date(2026, 1, 31)  # 13-month window end')
print()
print('KEY DIFFERENCES vs. current default (running today):')
print()

rows = [
    ['Parameter',           'Current Default (Jul 2026)',    'Historical Jan 2026 Run'],
    ['pi_EndScoreDt',       'None → all history (thru today)', 'date(2026, 1, 31)'],
    ['pi_BeginScoreDt',     'None → 1900-01-01',             'None (all history OK)'],
    ['pi_BeginRefFundDt',   f'{CURRENT_BEGIN_FUND_DT}',      f'{BEGIN_FUND_DT}'],
    ['pi_EndRefFundDt',     f'{CURRENT_END_FUND_DT}',        f'{END_FUND_DT}'],
]

df_params = pd.DataFrame(rows[1:], columns=rows[0])
print(df_params.to_string(index=False))
print()
print('NOTE: The single most critical change is setting pi_EndScoreDt = date(2026, 1, 31)')
print('      Without it, ModelCPR / ActualCPR / RecapScore will include Feb-Jun 2026')
print('      performance data, making the historical report stale.')

## 8. Check 7: Does `build_output_template` need a FundDt filter for historical runs?

`build_output_template` in the Python code pulls ALL-TIME unique metric values (States, FICO bands, etc.) from `vwOrigLoan` without a `FundDt` window filter. The SQL stored procedure also does this for most UNION branches. This means the template may include States/bands that had no loans in the specific 13-month window. Verify impact.

In [ ]:
# States seen ALL TIME vs states seen in Jan-2026 funding window
q_states_all = text("""
SELECT DISTINCT PropState
FROM ADM.ODS.vwOrigLoan a
WHERE a.Channel IN ('Correspondent', 'Co-Issue')
  AND a.Client IS NOT NULL
  AND a.FundDt IS NOT NULL
  AND a.FundedLoanAmt > 0
  AND a.MortgageType <> 'Reverse'
  AND NOT EXISTS (
      SELECT 1 FROM Analytics_Rpt.md.B2BPricingHist h
      WHERE h.LoanNum = a.LoanNum
        AND (h.ResolType = 'XFER' OR h.Channel = 'Bulk')
  )
""")

q_states_window = text("""
SELECT DISTINCT PropState
FROM ADM.ODS.vwOrigLoan a
WHERE a.Channel IN ('Correspondent', 'Co-Issue')
  AND a.Client IS NOT NULL
  AND a.FundDt BETWEEN :begin_dt AND :end_dt
  AND a.FundedLoanAmt > 0
  AND a.MortgageType <> 'Reverse'
  AND NOT EXISTS (
      SELECT 1 FROM Analytics_Rpt.md.B2BPricingHist h
      WHERE h.LoanNum = a.LoanNum
        AND (h.ResolType = 'XFER' OR h.Channel = 'Bulk')
  )
""")

states_all    = set(pd.read_sql(q_states_all,    engine)['PropState'].dropna())
states_window = set(pd.read_sql(q_states_window, engine,
                                params={'begin_dt': BEGIN_FUND_DT, 'end_dt': END_FUND_DT})['PropState'].dropna())

extra_states = states_all - states_window  # in template but NOT in the window

print(f'States in output template (all-time)         : {len(states_all)}')
print(f'States with loans in Jan-2026 fund window    : {len(states_window)}')
print(f'Extra template rows (zero volume in window)  : {len(extra_states)}')
if extra_states:
    print(f'  States: {sorted(extra_states)}')
    print()
    print('These states will appear in the report with 0 loan count / 0 funded amt')
    print('(consistent with SQL stored procedure behaviour — both use all-time template)')
else:
    print('No extra states — all template states have loans in the window.')

## 9. Summary & Conclusions

In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════╗
║         DIAGNOSTIC SUMMARY — Jan 31, 2026 Historical Run        ║
╠══════════════════════════════════════════════════════════════════╣
║                                                                  ║
║  1. B2BPricingHist availability                                  ║
║     → See Section 2 above for AsofDt range                      ║
║                                                                  ║
║  2. Funding window (Volume side)                                 ║
║     → Works correctly with explicit begin/end_fund_dt            ║
║                                                                  ║
║  3. Performance contamination (CRITICAL gap)                     ║
║     → Current default (pi_EndScoreDt = None) will pull          ║
║       B2BPricingHist rows AFTER Jan 2026 into CPR/Recap          ║
║     → Fix: set pi_EndScoreDt = date(2026, 1, 31)                ║
║                                                                  ║
║  4. No code changes needed — only parameter changes              ║
║     in B2B_Reporting_monthly_report.ipynb cell 6                 ║
║                                                                  ║
║  5. OutputTemplate (band/state list) uses all-time values         ║
║     → Same behaviour as the SQL stored procedure                 ║
║     → Zero-volume rows appear for inactive states/bands          ║
║                                                                  ║
╚══════════════════════════════════════════════════════════════════╝
""")